In [ ]:
!rm -rf nar-experiments
!git clone https://github.com/MarkoMile/nar-experiments.git
%cd nar-experiments
%pip install --no-cache-dir torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129
%pip install --no-cache-dir torch-scatter -f https://data.pyg.org/whl/torch-2.8.0+cu129.html
%pip install --no-cache-dir pyg-lib -f https://data.pyg.org/whl/torch-2.8.0+cu129.html
%pip install --no-cache-dir tensorflow
%pip install --no-cache-dir torch-geometric
%pip install --no-cache-dir lightning
%pip install --no-cache-dir ogb yacs loguru wandb
%pip install --no-cache-dir "salsa-clrs @ git+https://github.com/jkminder/SALSA-CLRS.git"
%pip install --no-cache-dir "dm-clrs @ git+https://github.com/deepmind/clrs.git"
%pip install --no-cache-dir -q numpy scipy networkx ogb matplotlib tqdm scikit-learn

In [ ]:
import tqdm.notebook
import tqdm.auto

# 1. Force the auto-detector to use the notebook version
tqdm.auto.tqdm = tqdm.notebook.tqdm
tqdm.auto.trange = tqdm.notebook.trange

In [ ]:
import os
import wandb
import getpass

# Optimize Python execution (removes asserts and __debug__ code)
os.environ["PYTHONOPTIMIZE"] = "1"

# Enable the new Rust-based core engine to prevent teardown hangs
os.environ["WANDB_CORE"] = "1"

# 2. Force Python OS-level output to be unbuffered
os.environ["PYTHONUNBUFFERED"] = "1"

# This will securely prompt you for your API key if it's not already set
if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass(prompt="Enter your WANDB_API_KEY: ")

try:
    wandb.login()
    print("WandB logged in successfully.")
except Exception as e:
    print(f"WandB login failed: {e}")

In [ ]:
import sys
import subprocess

config_path = 'src/configs/bfs/bestmodel.yml'
seeds = [43, 44, 45, 46, 47]

print(f"Running multi-seed evaluation on {config_path} for seeds: {seeds}")

for seed in seeds:
    print(f"\n{'='*50}\nStarting run for seed {seed}\n{'='*50}\n")
    cmd = [sys.executable, "-u", "src/experiments/train_bfs.py", 
           "--cfg", config_path, 
           "--seed", str(seed), 
           "--enable-wandb"]
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Run for seed {seed} failed with error: {e}")
        pass

In [ ]:
!vastai set api-key <api_key>
!vastai stop instance $CONTAINER_ID